# Y-Net Training and Test Evaluation

Run the notebook from top to bottom to train Y-Net, predict the test images, and calculate metrics. The saved 12-case comparison masks are read as a separate set; this notebook does not create replacements or export CSV/Excel files.


In [ ]:
from pathlib import Path
import sys, random
import numpy as np
import pandas as pd
import torch
from scipy import ndimage
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from PIL import Image
from IPython.display import display

def remove_small_srf(mask, min_pixels=10):
    labels, n = ndimage.label(mask == 2, structure=np.ones((3,3), dtype=np.uint8))
    sizes = np.bincount(labels.ravel())
    remove = np.zeros(n + 1, dtype=bool); remove[1:] = sizes[1:] < min_pixels
    mask[remove[labels]] = 0
    return mask

PROJECT = Path.cwd()
if PROJECT.name == "Code": PROJECT = PROJECT.parent
PACKAGE = PROJECT
DATA = PACKAGE / "Data"
REFERENCE = DATA / "test" / "masking_srf_reclassified"
REPLACED = DATA / "test" / "predictions_replaced_12_from_prior"
MODEL_CODE = PROJECT / "ynet_dataset_split_specific_20261109"
sys.path.insert(0, str(MODEL_CODE))
from ynet_model import YNet_general

PALETTE = np.array([[0,0,0], [255,0,0], [0,255,0], [255,255,255]], dtype=np.uint8)
CLASSES = {"ONL": 1, "SRF": 2, "EZ-RPE": 3}
REGIONS = {"512": (0,512), "중앙 1 mm": (216,296)}
SCALE_UM = 200 / 51
PIXEL_AREA_MM2 = 12.5 * SCALE_UM / 1_000_000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("장치:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU")
print("Train / val / test:", 200, 33, 68)


## 1. Data preparation

Load the training, validation, and test images and masks.

In [ ]:
def crop_rgb(a):
    if a.shape[0] == 496:
        a = np.pad(a, ((8,8),(0,0),(0,0)), constant_values=0)
    if a.shape[:2] != (512,768):
        raise ValueError(f"예상하지 않은 입력 크기: {a.shape}")
    return a[:,128:640]

def label_image(path):
    m = crop_rgb(np.asarray(Image.open(path).convert("RGB")))
    y = np.full(m.shape[:2], 255, dtype=np.uint8)
    for i, color in enumerate(PALETTE): y[np.all(m == color, axis=2)] = i
    if np.any(y == 255): raise ValueError(f"마스크 팔레트 오류: {path}")
    return y

def load_xy(split, mask_folder):
    root = DATA / split
    origins = {p.stem: p for p in (root/"origin").glob("*") if p.is_file()}
    masks = {p.stem: p for p in (root/mask_folder).glob("*") if p.is_file()}
    if origins.keys() != masks.keys(): raise ValueError(f"{split}: OCT/마스크 파일명이 일치하지 않습니다")
    names = sorted(origins)
    xs = [torch.from_numpy(crop_rgb(np.asarray(Image.open(origins[n]).convert("RGB"))).copy()).permute(2,0,1) for n in names]
    ys = [torch.from_numpy(label_image(masks[n])) for n in names]
    return TensorDataset(torch.stack(xs), torch.stack(ys)), names

train_set, train_names = load_xy("train", "masking")
val_set, val_names = load_xy("val", "masking")
test_root = DATA / "test" / "origin"
test_names = sorted(p.name for p in test_root.glob("*.tif"))
print(f"Loaded: train={len(train_names)}, val={len(val_names)}, test={len(test_names)}")


## 2. Y-Net training

Train the segmentation model.

In [ ]:
def seed_all(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def loss_fn(logits, target):
    ce = nn.functional.cross_entropy(logits, target)
    prob = logits.softmax(1)[:,1:]
    truth = nn.functional.one_hot(target, 4).permute(0,3,1,2)[:,1:].to(prob.dtype)
    inter = (prob*truth).sum((2,3)); den = prob.sum((2,3))+truth.sum((2,3))
    return ce + 1 - ((2*inter+1e-6)/(den+1e-6)).mean()

seed_all()
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
torch.set_num_threads(8)
model = YNet_general(out_channels=4).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.1, patience=5, threshold=1e-3, threshold_mode="rel")
generator = torch.Generator().manual_seed(42)
train_loader = DataLoader(train_set, batch_size=8, shuffle=True, generator=generator, num_workers=0, pin_memory=DEVICE.type=="cuda")
val_loader = DataLoader(val_set, batch_size=8, shuffle=False, num_workers=0, pin_memory=DEVICE.type=="cuda")
best_val, best_epoch, stale = float("inf"), 0, 0
best_weights = None
history = []
for epoch in range(1,101):
    model.train(); train_sum = train_n = 0
    for x,y in train_loader:
        x=x.to(DEVICE).float()/255; y=y.to(DEVICE).long()
        optimizer.zero_grad(set_to_none=True); loss=loss_fn(model(x),y); loss.backward(); optimizer.step()
        train_sum += loss.item()*len(x); train_n += len(x)
    model.eval(); val_sum = val_n = 0
    with torch.inference_mode():
        for x,y in val_loader:
            x=x.to(DEVICE).float()/255; y=y.to(DEVICE).long()
            val_sum += loss_fn(model(x),y).item()*len(x); val_n += len(x)
    train_loss, val_loss = train_sum/train_n, val_sum/val_n
    scheduler.step(val_loss)
    if val_loss < best_val:
        best_val, best_epoch, stale = val_loss, epoch, 0
        best_weights = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    else: stale += 1
    history.append({"epoch":epoch,"train_loss":train_loss,"val_loss":val_loss,"best_val_loss":best_val})
    print(f"{epoch:03d} | train {train_loss:.5f} | val {val_loss:.5f} | best {best_val:.5f} (epoch {best_epoch})")
    if stale >= 30: break
model.load_state_dict(best_weights)
model.eval()
print(f"학습 완료: {len(history)} epochs, best epoch={best_epoch}, best val loss={best_val:.5f}")


## 3. Test inference

Predict segmentation masks for the test images.

In [ ]:
test_x = torch.stack([torch.from_numpy(crop_rgb(np.asarray(Image.open(test_root/n).convert("RGB"))).copy()).permute(2,0,1) for n in test_names])
test_loader = DataLoader(TensorDataset(test_x), batch_size=8, shuffle=False, num_workers=0)
new_predictions = {}
with torch.inference_mode():
    offset = 0
    for (x,) in test_loader:
        pred = model(x.to(DEVICE).float()/255).argmax(1).cpu().numpy()
        for arr in pred: new_predictions[test_names[offset]] = arr.astype(np.uint8); offset += 1
print("새 모델 test 인퍼런스 완료:", len(new_predictions), "장")
new_predictions = {name: remove_small_srf(mask) for name, mask in new_predictions.items()}


## 4. Image-level metrics

Calculate Dice, thickness/MAE, and SRF area for new predictions and the saved 12-case comparison.

In [ ]:
def rgb_to_labels(rgb):
    out=np.full(rgb.shape[:2],255,dtype=np.uint8)
    for i,color in enumerate(PALETTE): out[np.all(rgb==color,axis=2)]=i
    if np.any(out==255): raise ValueError("알 수 없는 색이 포함된 마스크")
    return out

def dice(a,b):
    den=int(a.sum()+b.sum())
    return 2*int(np.count_nonzero(a&b))/den if den else 1.0

def measure_predictions(source, pred_map):
    result=[]
    for name in test_names:
        ref_rgb=crop_rgb(np.asarray(Image.open(REFERENCE/(Path(name).stem+".tif")).convert("RGB")))
        ref=rgb_to_labels(ref_rgb)
        pred=pred_map[name]
        group="SRF 있음" if np.any(ref==2) else "SRF 없음"
        for region,(x1,x2) in REGIONS.items():
            for label,cid in CLASSES.items():
                a=ref[:,x1:x2]==cid; b=pred[:,x1:x2]==cid
                row={"source":source,"image":name,"SRF_group":group,"region":region,"class":label,"Dice":dice(a,b)}
                if cid==2:
                    row["reference_area_mm2"]=a.sum()*PIXEL_AREA_MM2
                    row["prediction_area_mm2"]=b.sum()*PIXEL_AREA_MM2
                else:
                    row["reference_thickness_um"]=a.sum()/(x2-x1)*SCALE_UM
                    row["prediction_thickness_um"]=b.sum()/(x2-x1)*SCALE_UM
                    row["MAE_um"]=abs(row["reference_thickness_um"]-row["prediction_thickness_um"])
                result.append(row)
    return pd.DataFrame(result)

new_metrics = measure_predictions("새로 학습한 Y-Net", new_predictions)
replaced_predictions = {name: rgb_to_labels(np.asarray(Image.open(REPLACED/(Path(name).stem+".png")).convert("RGB"))) for name in test_names}
replaced_metrics = measure_predictions("기존 12장 대체 폴더", replaced_predictions)
all_metrics = pd.concat([new_metrics,replaced_metrics], ignore_index=True)
summary = (all_metrics.groupby(["source","SRF_group","region","class"], sort=False, dropna=False)
    .agg(n_images=("image","nunique"), mean_Dice=("Dice","mean"),
         reference_thickness_um=("reference_thickness_um","mean"), prediction_thickness_um=("prediction_thickness_um","mean"), MAE_um=("MAE_um","mean"),
         reference_SRF_area_mm2=("reference_area_mm2","mean"), prediction_SRF_area_mm2=("prediction_area_mm2","mean"))
    .reset_index())


## 5. Display

Show grouped summaries and image-level results.

In [ ]:
display(summary.round(4))
print("이미지별 점수:", len(all_metrics), "행")
display(all_metrics.round(4))
